# NGLView Visualisation

Create an interactive 3D visualisation. The Ig domain and TM helix will be highlighted using spacefill representation. Visual selections may need manual adjustment if NGLView has trouble interpreting Martini topology automatically.

In [ ]:
import nglview as nv
import ipywidgets
import numpy as np

# ---------------------------------------------------------
# Workaround for Martini bonds: Add POPC bonds explicitly
# so NGLView can render the membrane as transparent sticks (licorice)
# ---------------------------------------------------------
popc_bonds = []
popc_res = u.select_atoms("resname POPC").residues
for res in popc_res:
    idx = {name: atom.index for name, atom in zip(res.atoms.names, res.atoms)}
    pairs = [("NC3","PO4"), ("PO4","GL1"), ("GL1","GL2"),
             ("GL1","C1A"), ("C1A","D2A"), ("D2A","C3A"), ("C3A","C4A"),
             ("GL2","C1B"), ("C1B","C2B"), ("C2B","C3B"), ("C3B","C4B")]
    for a, b in pairs:
        if a in idx and b in idx:
            popc_bonds.append((idx[a], idx[b]))

if not hasattr(u, 'bonds') or len(u.bonds) == 0:
    u.add_TopologyAttr('bonds', popc_bonds)
else:
    # If there are existing bonds, just append them
    all_bonds = list(u.bonds.indices) + popc_bonds
    u.add_TopologyAttr('bonds', all_bonds)

# ---------------------------------------------------------
# Create precise NGLView selections using explicit atom indices
# to avoid any "floating objects" matching ambiguous ranges
# ---------------------------------------------------------
ig_sel = "@" + ",".join(map(str, ig_domain.indices))
tm_sel = "@" + ",".join(map(str, tm_helix.indices))
hinge_sel = "@" + ",".join(map(str, u.select_atoms("resid 90-107").indices))
rest_sel = "@" + ",".join(map(str, u.select_atoms("resid 1-2 129-138").indices))
popc_sel = "@" + ",".join(map(str, u.select_atoms("resname POPC").indices))

# ---------------------------------------------------------
# Setup Viewer
# ---------------------------------------------------------
view = nv.show_mdanalysis(u)
view.clear_representations()

# Color the cartoon (tube) representation per domain
view.add_representation("tube", selection=ig_sel, color="crimson", radius=0.6)
view.add_representation("tube", selection=hinge_sel, color="orange", radius=0.6)
view.add_representation("tube", selection=tm_sel, color="navy", radius=0.6)
if len(u.select_atoms("resid 1-2 129-138").indices) > 0:
    view.add_representation("tube", selection=rest_sel, color="lightgrey", radius=0.4)

# Add the POPC membrane as transparent sticks (licorice)
# Since we added bonds, NGLView will draw the sticks correctly!
view.add_representation("licorice", selection=popc_sel, color="silver", opacity=0.3)

# Rotate camera by -90 degrees around X axis to show membrane side-on
# and flip the Z axis so the IG domain is UPRIGHT (at the top)
view.control.rotate([-0.7071, 0, 0, 0.7071])

# Create a time counter label
time_label = ipywidgets.Label(value="Time: 0.0 ns")

# Define callback to update time counter when frame changes
def on_frame_change(change):
    frame = change['new']
    # Each frame is 1 ns based on dt=1000 ps
    time_ns = frame * 1.0 
    time_label.value = f"Time: {time_ns:.1f} ns"

view.observe(on_frame_change, names=['frame'])

# Display the viewer and the time counter together
display(ipywidgets.VBox([view, time_label]))